# Section 1: Runtime Setup

In [1]:
!pip install "numpy<2.0.0" -q
!pip install git+https://github.com/huggingface/diarizers.git -q
!pip install transformers datasets pyannote.audio torch torchaudio accelerate scipy pyannote.metrics -q
!pip install peft -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 42.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import os

BASE_DIR          = "/content/drive/MyDrive/ami_diarization_v2"
PREPROCESSED_DIR  = f"{BASE_DIR}/preprocessed_10s_ihm_v1"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints_head_only"
BEST_MODEL_PATH   = f"{CHECKPOINT_DIR}/best_model.pt"
BEST_META_PATH    = f"{CHECKPOINT_DIR}/best_model_metadata.json"
LOG_DIR           = f"{BASE_DIR}/logs"

HF_DATASET        = "diarizers-community/ami"
HF_CONFIG         = "ihm"
MODEL_ID          = "pyannote/segmentation-3.0"

SAMPLE_RATE             = 16000
CHUNK_DURATION          = 10
MAX_SPEAKERS_PER_CHUNK  = 4
MAX_SPEAKERS_PER_FRAME  = 2
TRAIN_NUM_SPEAKERS      = 7  # validated against model output in Section 4

EPOCHS        = 10
ACCUM_STEPS   = 16
LR            = 1e-3
WEIGHT_DECAY  = 1e-2
MAX_GRAD_NORM = 1.0
POS_WEIGHT    = 5.0

TRAINABLE_NAME_PATTERNS = ["classifier", "head", "linear"]

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Constants defined.")
print(f"  BASE_DIR:           {BASE_DIR}")
print(f"  PREPROCESSED_DIR:   {PREPROCESSED_DIR}")
print(f"  CHECKPOINT_DIR:     {CHECKPOINT_DIR}")
print(f"  MODEL_ID:           {MODEL_ID}")
print(f"  TRAIN_NUM_SPEAKERS: {TRAIN_NUM_SPEAKERS} (validated against model output in Section 4)")

Constants defined.
  BASE_DIR:           /content/drive/MyDrive/ami_diarization_v2
  PREPROCESSED_DIR:   /content/drive/MyDrive/ami_diarization_v2/preprocessed_10s_ihm_v1
  CHECKPOINT_DIR:     /content/drive/MyDrive/ami_diarization_v2/checkpoints_head_only
  MODEL_ID:           pyannote/segmentation-3.0
  TRAIN_NUM_SPEAKERS: 7 (validated against model output in Section 4)


# Section 2: Load Or Build Preprocessed Data

In [ ]:
import numpy as np
from datasets import load_from_disk

REQUIRED_COLUMNS = ["waveforms", "targets", "labels"]
SPLITS = ["train", "val", "test"]


def preprocessed_data_is_valid(base_path: str) -> bool:
    print(f"\nValidating preprocessed data at: {base_path}")
    ok = True

    for split in SPLITS:
        split_path = f"{base_path}/{split}"

        if not os.path.isdir(split_path):
            print(f"  [FAIL] {split}: directory not found")
            ok = False
            continue

        try:
            ds = load_from_disk(split_path)
        except Exception as e:
            print(f"  [FAIL] {split}: load_from_disk raised: {e}")
            ok = False
            continue

        if len(ds) == 0:
            print(f"  [FAIL] {split}: dataset is empty")
            ok = False
            continue

        missing_cols = [c for c in REQUIRED_COLUMNS if c not in ds.column_names]
        if missing_cols:
            print(f"  [FAIL] {split}: missing columns {missing_cols}")
            ok = False
            continue

        split_ok = True
        for i in range(min(3, len(ds))):
            item     = ds[i]
            waveform = np.array(item["waveforms"])
            target   = np.array(item["targets"])

            if waveform.size == 0:
                print(f"  [FAIL] {split}[{i}]: waveform is empty")
                split_ok = False; break
            if np.any(np.isnan(waveform)) or np.any(np.isinf(waveform)):
                print(f"  [FAIL] {split}[{i}]: waveform has NaN/Inf")
                split_ok = False; break
            if target.ndim != 2:
                print(f"  [FAIL] {split}[{i}]: target.ndim={target.ndim}, expected 2")
                split_ok = False; break
            if target.shape[0] == 0:
                print(f"  [FAIL] {split}[{i}]: target has zero frames, shape={target.shape}")
                split_ok = False; break
            if target.shape[1] > MAX_SPEAKERS_PER_CHUNK:
                print(f"  [FAIL] {split}[{i}]: target has {target.shape[1]} speakers > {MAX_SPEAKERS_PER_CHUNK}")
                split_ok = False; break
            if target.shape[1] == 0:
                print(f"  [FAIL] {split}[{i}]: target has no speaker columns (silence-only chunk leaked into dataset), shape={target.shape}")
                split_ok = False; break
            if np.any(np.isnan(target)) or np.any(np.isinf(target)):
                print(f"  [FAIL] {split}[{i}]: target has NaN/Inf")
                split_ok = False; break

        if not split_ok:
            ok = False
        else:
            ex_w = np.array(ds[0]["waveforms"])
            ex_t = np.array(ds[0]["targets"])
            print(f"  [OK]   {split}: {len(ds)} chunks  waveform={ex_w.shape}  target={ex_t.shape}")

    if ok:
        print("Validation PASSED. Reusing existing preprocessed data.")
    else:
        print("Validation FAILED. Will rebuild preprocessed data.")
    return ok

In [ ]:
import json
import shutil
from datasets import load_dataset, Audio, Dataset, concatenate_datasets
from diarizers import Preprocess, SegmentationModelConfig


def build_preprocessing(save_dir: str):
    print(f"\nBuilding preprocessed data into: {save_dir}")
    os.makedirs(save_dir, exist_ok=True)

    raw = load_dataset(HF_DATASET, HF_CONFIG)
    raw = raw.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE, decode=True))

    config = SegmentationModelConfig(
        chunk_duration=CHUNK_DURATION,
        max_speakers_per_chunk=MAX_SPEAKERS_PER_CHUNK,
        max_speakers_per_frame=MAX_SPEAKERS_PER_FRAME,
    )
    preprocessor = Preprocess(config)

    def safe_file(row):
        audio = row["audio"]
        arr   = audio["array"] if isinstance(audio["array"], np.ndarray) else np.array(audio["array"])
        return {
            "audio":            [{"array": arr, "sampling_rate": SAMPLE_RATE}],
            "timestamps_start": [row["timestamps_start"]],
            "timestamps_end":   [row["timestamps_end"]],
            "speakers":         [row["speakers"]],
        }

    split_map = {"train": "train", "validation": "val", "test": "test"}
    BATCH_SIZE = 5
    WORK_DIR   = "/content/preprocessed_work"

    for raw_split, out_split in split_map.items():
        if raw_split not in raw:
            print(f"  WARNING: split '{raw_split}' not found in dataset")
            continue

        data = raw[raw_split]
        print(f"  Processing {raw_split} -> {out_split} ({len(data)} files)...")
        work_split = f"{WORK_DIR}/{out_split}"
        os.makedirs(work_split, exist_ok=True)

        batch_paths = []
        batch_w, batch_t, batch_l = [], [], []
        batch_idx    = 0
        skipped_zero = 0

        for i, row in enumerate(data):
            file = safe_file(row)
            for start_time in preprocessor.get_start_positions(file, overlap=0.0):
                waveform, y, labels = preprocessor.get_chunk(file, start_time)
                if y.shape[1] == 0:
                    skipped_zero += 1
                    continue
                batch_w.append(waveform)
                batch_t.append(y)
                batch_l.append(labels)

            if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(data):
                if batch_w:
                    bds  = Dataset.from_dict({"waveforms": batch_w, "targets": batch_t, "labels": batch_l})
                    bpath = f"{work_split}/batch_{batch_idx:04d}"
                    bds.save_to_disk(bpath)
                    batch_paths.append(bpath)
                    print(f"    batch {batch_idx}: {len(batch_w)} chunks")
                    batch_w, batch_t, batch_l = [], [], []
                    batch_idx += 1

        if skipped_zero > 0:
            print(f"  [INFO] {out_split}: skipped {skipped_zero} silence-only chunks (no speaker annotations)")

        merged = concatenate_datasets([load_from_disk(p) for p in batch_paths])
        merged.save_to_disk(f"{save_dir}/{out_split}")
        print(f"  Saved {len(merged)} chunks -> {save_dir}/{out_split}")

    meta = {
        "dataset":                    HF_DATASET,
        "config":                     HF_CONFIG,
        "sample_rate":                SAMPLE_RATE,
        "chunk_duration":             CHUNK_DURATION,
        "max_speakers_per_chunk":     MAX_SPEAKERS_PER_CHUNK,
        "max_speakers_per_frame":     MAX_SPEAKERS_PER_FRAME,
        "created_by_notebook_version": "ami_diarization_v2",
        "columns":                    ["waveforms", "targets", "labels"],
    }
    with open(f"{save_dir}/metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    shutil.rmtree(WORK_DIR, ignore_errors=True)
    print("Build complete.")


if preprocessed_data_is_valid(PREPROCESSED_DIR):
    train_ds = load_from_disk(f"{PREPROCESSED_DIR}/train")
    val_ds   = load_from_disk(f"{PREPROCESSED_DIR}/val")
    test_ds  = load_from_disk(f"{PREPROCESSED_DIR}/test")
else:
    build_preprocessing(PREPROCESSED_DIR)
    train_ds = load_from_disk(f"{PREPROCESSED_DIR}/train")
    val_ds   = load_from_disk(f"{PREPROCESSED_DIR}/val")
    test_ds  = load_from_disk(f"{PREPROCESSED_DIR}/test")

print(f"\nData ready: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

# Section 3: Dataset And Collator

In [6]:
import torch
import numpy as np
from torch.utils.data import Dataset as TorchDataset, DataLoader


class DiarizationDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            "waveforms": torch.tensor(item["waveforms"], dtype=torch.float32),
            "targets":   torch.tensor(np.array(item["targets"]), dtype=torch.float32),
        }


class DiarizationCollator:
    """
    Pads/trims speaker dimension to num_speakers.
    When an example has more speakers than num_speakers, keeps the most active ones.
    Stops loudly if a target is not 2D.
    """
    def __init__(self, num_speakers: int):
        self.num_speakers = num_speakers

    def __call__(self, features):
        waveforms      = torch.stack([f["waveforms"] for f in features])
        padded_targets = []

        for f in features:
            t = f["targets"].numpy()  # [frames, speakers]
            assert t.ndim == 2, f"Expected 2D target, got {t.ndim}D"
            n = t.shape[1]

            if n > self.num_speakers:
                top = np.argsort(-t.sum(axis=0))[:self.num_speakers]
                t   = t[:, top]
            elif n < self.num_speakers:
                pad = np.zeros((t.shape[0], self.num_speakers - n), dtype=t.dtype)
                t   = np.concatenate([t, pad], axis=1)

            padded_targets.append(t)

        return {
            "waveforms": waveforms,
            "targets":   torch.tensor(np.stack(padded_targets), dtype=torch.float32),
        }


collator    = DiarizationCollator(num_speakers=TRAIN_NUM_SPEAKERS)
train_torch = DiarizationDataset(train_ds)
val_torch   = DiarizationDataset(val_ds)
test_torch  = DiarizationDataset(test_ds)

train_loader = DataLoader(train_torch, batch_size=1, shuffle=True,  collate_fn=collator, num_workers=0)
val_loader   = DataLoader(val_torch,   batch_size=1, shuffle=False, collate_fn=collator, num_workers=0)
test_loader  = DataLoader(test_torch,  batch_size=1, shuffle=False, collate_fn=collator, num_workers=0)

print("DataLoaders ready.")
print(f"  train: {len(train_torch)}  val: {len(val_torch)}  test: {len(test_torch)}")

DataLoaders ready.
  train: 28974  val: 3473  test: 3254


# Section 4: Model Loading And Shape Probe

In [7]:
import functools
from pyannote.audio import Model as PyanModel

# Patch torch.load for Colab/PyTorch compatibility
_orig_torch_load = torch.serialization.load

@functools.wraps(_orig_torch_load)
def _patched_load(f, *args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_torch_load(f, *args, **kwargs)

torch.load = _patched_load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected, using CPU")

model = PyanModel.from_pretrained(MODEL_ID)
model = model.to(device)
print(f"Model loaded: {MODEL_ID}")

# Shape probe: one forward pass before any training or freezing
probe_batch  = next(iter(train_loader))
probe_wave   = probe_batch["waveforms"].to(device)
probe_target = probe_batch["targets"]

model.eval()
with torch.no_grad():
    probe_out = model(probe_wave)

print(f"\nShape probe:")
print(f"  waveforms shape: {probe_wave.shape}")
print(f"  targets shape:   {probe_target.shape}")
print(f"  outputs shape:   {probe_out.shape}")

model_spk  = probe_out.shape[-1]
target_spk = probe_target.shape[-1]

if model_spk != target_spk:
    raise RuntimeError(
        f"SPEAKER DIMENSION MISMATCH: "
        f"model outputs {model_spk} channels but targets have {target_spk} speakers. "
        f"Set TRAIN_NUM_SPEAKERS={model_spk} in Section 1 and rerun."
    )

print(f"\nSpeaker dimension check PASSED: model={model_spk}, targets={target_spk}")

GPU: Tesla T4


pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

Model loaded: pyannote/segmentation-3.0


/usr/local/lib/python3.12/dist-packages/asteroid_filterbanks/enc_dec.py:202: UserWarning: Input tensor was 2D. Applying the corresponding Decoder to the current output will result in a 3D tensor. This behaviours was introduced to match Conv1D and ConvTranspose1D, please use 3D inputs to avoid it. For example, this can be done with input_tensor.unsqueeze(1).
  warnings.warn(



Shape probe:
  waveforms shape: torch.Size([1, 160000])
  targets shape:   torch.Size([1, 589, 7])
  outputs shape:   torch.Size([1, 589, 7])

Speaker dimension check PASSED: model=7, targets=7


# Section 5: PIT Loss

In [8]:
import torch.nn as nn
from scipy.optimize import linear_sum_assignment


def pit_bce_loss(logits: torch.Tensor, targets: torch.Tensor, pos_weight: float = POS_WEIGHT) -> torch.Tensor:
    """
    Permutation-Invariant Training loss using Hungarian matching.

    logits:  [batch, frames, speakers]
    targets: [batch, frames, speakers]
    Returns scalar mean loss over batch.
    """
    batch_size, num_frames, num_speakers = logits.shape
    pw        = torch.tensor([pos_weight], device=logits.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw, reduction="none")
    losses    = []

    for b in range(batch_size):
        # Build cost matrix without gradients (only used for matching)
        with torch.no_grad():
            cost = torch.zeros(num_speakers, num_speakers, device=logits.device)
            for i in range(num_speakers):
                for j in range(num_speakers):
                    cost[i, j] = criterion(logits[b, :, i], targets[b, :, j]).mean()

        row_ind, col_ind = linear_sum_assignment(cost.cpu().numpy())

        # Differentiable loss on the optimal assignment
        matched_loss = torch.tensor(0.0, device=logits.device)
        for i, j in zip(row_ind, col_ind):
            matched_loss = matched_loss + criterion(logits[b, :, i], targets[b, :, j]).mean()

        losses.append(matched_loss / num_speakers)

    return torch.stack(losses).mean()


# Sanity check: loss must be scalar and backward must work
model.train()
_t    = probe_target.to(device)
_out  = model(probe_wave)
_mf   = min(_out.shape[1], _t.shape[1])
_loss = pit_bce_loss(_out[:, :_mf, :], _t[:, :_mf, :])
_loss.backward()
print(f"PIT loss sanity check PASSED. loss={_loss.item():.4f}")
model.zero_grad()
del _t, _out, _loss

PIT loss sanity check PASSED. loss=0.7769


# Section 6: Parameter-Efficient Fine-Tuning Strategy

In [9]:
def apply_freezing(mdl, patterns):
    for p in mdl.parameters():
        p.requires_grad = False
    for name, p in mdl.named_parameters():
        if any(pat in name.lower() for pat in patterns):
            p.requires_grad = True


def trainability_report(mdl):
    total     = sum(p.numel() for p in mdl.parameters())
    trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    pct       = 100.0 * trainable / total if total > 0 else 0.0
    names     = [n for n, p in mdl.named_parameters() if p.requires_grad]

    print(f"\nParameter trainability:")
    print(f"  Total:      {total:,}")
    print(f"  Trainable:  {trainable:,}")
    print(f"  Percent:    {pct:.2f}%")
    print(f"  Trainable layers ({len(names)}):")
    for n in names:
        print(f"    {n}")

    return total, trainable, pct


# Print candidate modules to inform pattern selection
CANDIDATE_KEYWORDS = ["classifier", "linear", "projection", "proj", "head", "output"]
print("Candidate modules (inspect to adjust TRAINABLE_NAME_PATTERNS if needed):")
for name, module in model.named_modules():
    if any(kw in name.lower() for kw in CANDIDATE_KEYWORDS):
        print(f"  {name:60s}  {type(module).__name__}")

# Reload fresh model so freezing starts from clean state
model = PyanModel.from_pretrained(MODEL_ID).to(device)

apply_freezing(model, TRAINABLE_NAME_PATTERNS)
total_params, trainable_params, trainable_pct = trainability_report(model)

if trainable_params == 0:
    raise RuntimeError(
        f"No trainable parameters matched patterns {TRAINABLE_NAME_PATTERNS}. "
        "Inspect the candidate list above and update TRAINABLE_NAME_PATTERNS."
    )

if trainable_pct > 10.0:
    raise RuntimeError(
        f"Trainable percent {trainable_pct:.2f}% exceeds 10%. "
        "Tighten TRAINABLE_NAME_PATTERNS to freeze more layers."
    )

print(f"\nFreeze check PASSED: {trainable_pct:.2f}% trainable (target 1-5%)")

Candidate modules (inspect to adjust TRAINABLE_NAME_PATTERNS if needed):
  linear                                                        ModuleList
  linear.0                                                      Linear
  linear.1                                                      Linear
  classifier                                                    Linear

Parameter trainability:
  Total:      1,473,265
  Trainable:  50,311
  Percent:    3.41%
  Trainable layers (6):
    linear.0.weight
    linear.0.bias
    linear.1.weight
    linear.1.bias
    classifier.weight
    classifier.bias

Freeze check PASSED: 3.41% trainable (target 1-5%)


# Section 7: Optional LoRA Exploration

This cell is **diagnostic only** and does not affect the main training path, which uses frozen-backbone/head-only fine-tuning.

In [10]:
linear_modules = [name for name, m in model.named_modules() if isinstance(m, torch.nn.Linear)]
print(f"Found {len(linear_modules)} nn.Linear modules:")
for n in linear_modules:
    print(f"  {n}")

if len(linear_modules) >= 4:
    print("\nAttempting LoRA compatibility check...")
    try:
        from peft import LoraConfig, get_peft_model
        lora_cfg   = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=linear_modules[:4],
            lora_dropout=0.05,
            bias="none",
        )
        lora_probe = get_peft_model(model, lora_cfg)
        lora_probe.print_trainable_parameters()
        del lora_probe
        print("LoRA COMPATIBLE. To use LoRA, replace model with the lora_probe above.")
    except Exception as e:
        print(f"LoRA NOT compatible: {e}")
        print("Continuing with frozen-backbone/head-only fine-tuning.")
else:
    print("Too few Linear modules for useful LoRA. Using head-only fine-tuning.")

Found 3 nn.Linear modules:
  linear.0
  linear.1
  classifier
Too few Linear modules for useful LoRA. Using head-only fine-tuning.


# Section 8: Optimizer And Scheduler

In [11]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

trainable_param_list = [p for p in model.parameters() if p.requires_grad]
assert len(trainable_param_list) > 0, "No trainable parameters available for optimizer."

optimizer = AdamW(trainable_param_list, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Optimizer: AdamW over {len(trainable_param_list)} trainable parameter tensors")
print(f"  LR={LR}, weight_decay={WEIGHT_DECAY}")
print(f"Scheduler: CosineAnnealingLR, T_max={EPOCHS}")

Optimizer: AdamW over 6 trainable parameter tensors
  LR=0.001, weight_decay=0.01
Scheduler: CosineAnnealingLR, T_max=10


# Section 9: Resumable Checkpointing Setup

In [12]:
import glob
import re


def get_latest_checkpoint(ckpt_dir: str):
    files = glob.glob(os.path.join(ckpt_dir, "epoch_*.pt"))
    if not files:
        return None
    def _epoch_num(path):
        m = re.search(r"epoch_(\d+)\.pt$", path)
        return int(m.group(1)) if m else -1
    return max(files, key=_epoch_num)


def save_checkpoint(path, epoch, mdl, opt, sched, train_loss, val_loss, best_val):
    torch.save({
        "epoch":                  epoch,
        "model_state_dict":       mdl.state_dict(),
        "optimizer_state_dict":   opt.state_dict(),
        "scheduler_state_dict":   sched.state_dict(),
        "train_loss":             train_loss,
        "val_loss":               val_loss,
        "best_val_loss":          best_val,
        "trainable_name_patterns": TRAINABLE_NAME_PATTERNS,
        "trainable_param_count":  trainable_params,
        "total_param_count":      total_params,
        "config": {
            "model_id":           MODEL_ID,
            "epochs":             EPOCHS,
            "lr":                 LR,
            "weight_decay":       WEIGHT_DECAY,
            "accum_steps":        ACCUM_STEPS,
            "max_grad_norm":      MAX_GRAD_NORM,
            "train_num_speakers": TRAIN_NUM_SPEAKERS,
            "pos_weight":         POS_WEIGHT,
        },
    }, path)


START_EPOCH   = 0
best_val_loss = float("inf")
best_epoch    = 0
latest_ckpt   = get_latest_checkpoint(CHECKPOINT_DIR)

if latest_ckpt:
    print(f"Resuming from: {latest_ckpt}")
    ckpt = torch.load(latest_ckpt, weights_only=False, map_location=device)

    # Re-apply freezing before loading optimizer state
    apply_freezing(model, TRAINABLE_NAME_PATTERNS)
    model.load_state_dict(ckpt["model_state_dict"])

    # Recreate optimizer from current trainable parameters
    trainable_param_list = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_param_list, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])

    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)

    START_EPOCH   = ckpt["epoch"]
    best_val_loss = ckpt["best_val_loss"]
    print(f"  Resumed from epoch {START_EPOCH}, best_val_loss={best_val_loss:.4f}")
else:
    print("No checkpoint found. Starting from epoch 0.")

No checkpoint found. Starting from epoch 0.


# Sections 10 & 11: Training Loop With Diagnostics

In [13]:
import time

def format_time(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}h {m:02d}m {s:02d}s"


print("=" * 60)
print("TRAINING STARTUP DIAGNOSTICS")
print("=" * 60)
print(f"Device:               {device}")
if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")
print(f"Model ID:             {MODEL_ID}")
print(f"Train chunks:         {len(train_torch)}")
print(f"Val chunks:           {len(val_torch)}")
print(f"Test chunks:          {len(test_torch)}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percent:    {trainable_pct:.2f}%")
print(f"Trainable patterns:   {TRAINABLE_NAME_PATTERNS}")
print(f"Example waveform:     {probe_wave.shape}")
print(f"Example target:       {probe_target.shape}")
print(f"Example output:       {probe_out.shape}")

model.eval()
with torch.no_grad():
    _io  = model(probe_wave.to(device))
    _it  = probe_target.to(device)
    _imf = min(_io.shape[1], _it.shape[1])
    _il  = pit_bce_loss(_io[:, :_imf, :], _it[:, :_imf, :])
print(f"Initial PIT loss:     {_il.item():.4f}")
print(f"Start epoch:          {START_EPOCH}")
print("=" * 60)

TRAINING STARTUP DIAGNOSTICS
Device:               cuda
GPU:                  Tesla T4
Model ID:             pyannote/segmentation-3.0
Train chunks:         28974
Val chunks:           3473
Test chunks:          3254
Total parameters:     1,473,265
Trainable parameters: 50,311
Trainable percent:    3.41%
Trainable patterns:   ['classifier', 'head', 'linear']
Example waveform:     torch.Size([1, 160000])
Example target:       torch.Size([1, 589, 7])
Example output:       torch.Size([1, 589, 7])
Initial PIT loss:     0.7485
Start epoch:          0


In [ ]:
from tqdm.auto import tqdm


PROGRESS_METRIC_THRESHOLD = 0.5
PROGRESS_EPS              = 1e-8


def new_metric_totals():
    return {"correct": 0, "total": 0, "tp": 0, "fp": 0, "fn": 0}


def update_pit_metric_totals(totals, logits, targets, threshold=PROGRESS_METRIC_THRESHOLD):
    """
    Update frame-level metrics after matching predicted speaker streams to target
    streams with the same Hungarian assignment used by the PIT loss.
    """
    batch_size, num_frames, num_speakers = logits.shape
    pw        = torch.tensor([POS_WEIGHT], device=logits.device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw, reduction="none")

    with torch.no_grad():
        probs = torch.sigmoid(logits)
        for b in range(batch_size):
            cost = torch.zeros(num_speakers, num_speakers, device=logits.device)
            for i in range(num_speakers):
                for j in range(num_speakers):
                    cost[i, j] = criterion(logits[b, :, i], targets[b, :, j]).mean()

            row_ind, col_ind = linear_sum_assignment(cost.cpu().numpy())
            pred = probs[b, :, list(row_ind)] >= threshold
            true = targets[b, :, list(col_ind)] >= 0.5

            totals["correct"] += (pred == true).sum().item()
            totals["total"]   += true.numel()
            totals["tp"]      += (pred & true).sum().item()
            totals["fp"]      += (pred & ~true).sum().item()
            totals["fn"]      += (~pred & true).sum().item()

    return totals


def summarize_metric_totals(totals):
    accuracy  = totals["correct"] / max(totals["total"], 1)
    precision = totals["tp"] / max(totals["tp"] + totals["fp"], 1)
    recall    = totals["tp"] / max(totals["tp"] + totals["fn"], 1)
    f1        = 2 * precision * recall / max(precision + recall, PROGRESS_EPS)
    return {
        "accuracy":  accuracy,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
    }


def estimate_eta(done_units, total_units, elapsed):
    if done_units <= 0 or elapsed <= 0:
        return "calculating"
    remaining_units = max(total_units - done_units, 0)
    return format_time((elapsed / done_units) * remaining_units)


frame_mismatch_warnings = 0
training_start          = time.time()
train_batches           = len(train_loader)
val_batches             = len(val_loader)
epoch_units             = train_batches + val_batches
total_units             = max((EPOCHS - START_EPOCH) * epoch_units, 1)
completed_units         = 0

print(f"Training epochs {START_EPOCH + 1} to {EPOCHS}...")
print(f"Progress: {train_batches} train batches + {val_batches} val batches per epoch")
print("Metrics use PIT-aligned frame/speaker predictions at threshold "
      f"{PROGRESS_METRIC_THRESHOLD:.2f}.")

for epoch in range(START_EPOCH, EPOCHS):
    epoch_num    = epoch + 1
    epoch_start  = time.time()
    model.train()
    train_loss   = 0.0
    train_metrics = new_metric_totals()
    steps        = 0
    optimizer.zero_grad()

    train_bar = tqdm(
        train_loader,
        total=train_batches,
        desc=f"Epoch {epoch_num:03d}/{EPOCHS} train",
        unit="batch",
        leave=False,
    )

    for step, batch in enumerate(train_bar):
        waveforms  = batch["waveforms"].to(device)
        targets    = batch["targets"].to(device)
        outputs    = model(waveforms)
        min_frames = min(outputs.shape[1], targets.shape[1])
        logits     = outputs[:, :min_frames, :]
        targets    = targets[:, :min_frames, :]

        if outputs.shape[1] != batch["targets"].shape[1]:
            frame_mismatch_warnings += 1
            if frame_mismatch_warnings <= 5:
                print(f"  [warn] frame mismatch step {step}: out={outputs.shape[1]} tgt={batch['targets'].shape[1]}, using min={min_frames}")

        loss = pit_bce_loss(logits, targets)
        (loss / ACCUM_STEPS).backward()
        train_loss += loss.item()
        steps      += 1
        update_pit_metric_totals(train_metrics, logits.detach(), targets.detach())

        if (step + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_param_list, MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()

        done_units = completed_units + steps
        metrics    = summarize_metric_totals(train_metrics)
        train_bar.set_postfix({
            "loss":      f"{train_loss / max(steps, 1):.4f}",
            "acc":       f"{metrics['accuracy']:.3f}",
            "f1":        f"{metrics['f1']:.3f}",
            "epoch_eta": estimate_eta(steps, epoch_units, time.time() - epoch_start),
            "total_eta": estimate_eta(done_units, total_units, time.time() - training_start),
        })

    # Step for any remaining accumulated gradients
    if steps % ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(trainable_param_list, MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad()

    scheduler.step()
    avg_train     = train_loss / max(steps, 1)
    train_summary = summarize_metric_totals(train_metrics)
    completed_units += train_batches

    # Validation
    model.eval()
    val_loss    = 0.0
    val_metrics = new_metric_totals()
    val_steps   = 0

    val_bar = tqdm(
        val_loader,
        total=val_batches,
        desc=f"Epoch {epoch_num:03d}/{EPOCHS} val",
        unit="batch",
        leave=False,
    )

    with torch.no_grad():
        for batch in val_bar:
            waveforms  = batch["waveforms"].to(device)
            targets    = batch["targets"].to(device)
            outputs    = model(waveforms)
            min_frames = min(outputs.shape[1], targets.shape[1])
            logits     = outputs[:, :min_frames, :]
            targets    = targets[:, :min_frames, :]
            loss       = pit_bce_loss(logits, targets)

            val_loss  += loss.item()
            val_steps += 1
            update_pit_metric_totals(val_metrics, logits, targets)

            done_units = completed_units + val_steps
            metrics    = summarize_metric_totals(val_metrics)
            val_bar.set_postfix({
                "loss":      f"{val_loss / max(val_steps, 1):.4f}",
                "acc":       f"{metrics['accuracy']:.3f}",
                "f1":        f"{metrics['f1']:.3f}",
                "epoch_eta": estimate_eta(train_batches + val_steps, epoch_units, time.time() - epoch_start),
                "total_eta": estimate_eta(done_units, total_units, time.time() - training_start),
            })

    avg_val     = val_loss / max(val_steps, 1)
    val_summary = summarize_metric_totals(val_metrics)
    completed_units += val_batches
    epoch_time  = time.time() - epoch_start
    total_eta   = estimate_eta(completed_units, total_units, time.time() - training_start)

    print(
        f"Epoch {epoch_num:03d}/{EPOCHS} "
        f"train_loss={avg_train:.4f} train_acc={train_summary['accuracy']:.3f} train_f1={train_summary['f1']:.3f} "
        f"val_loss={avg_val:.4f} val_acc={val_summary['accuracy']:.3f} val_f1={val_summary['f1']:.3f} "
        f"time={format_time(epoch_time)} total_eta={total_eta} best_val={best_val_loss:.4f}"
    )
    print(
        f"  Train P/R/F1: {train_summary['precision']:.3f}/{train_summary['recall']:.3f}/{train_summary['f1']:.3f} | "
        f"Val P/R/F1: {val_summary['precision']:.3f}/{val_summary['recall']:.3f}/{val_summary['f1']:.3f}"
    )

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch_num:03d}.pt")
    save_checkpoint(ckpt_path, epoch_num, model, optimizer, scheduler, avg_train, avg_val, best_val_loss)
    print(f"  Checkpoint: {ckpt_path}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_epoch    = epoch_num
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        with open(BEST_META_PATH, "w") as f:
            json.dump({
                "epoch":             best_epoch,
                "train_loss":        avg_train,
                "val_loss":          avg_val,
                "train_accuracy":    train_summary["accuracy"],
                "train_f1":          train_summary["f1"],
                "val_accuracy":      val_summary["accuracy"],
                "val_f1":            val_summary["f1"],
                "metric_threshold":  PROGRESS_METRIC_THRESHOLD,
                "trainable_percent": trainable_pct,
            }, f, indent=2)
        print(f"  Best model updated (epoch {best_epoch}, val={best_val_loss:.4f})")

total_time = time.time() - training_start
print("=" * 60)
print("TRAINING COMPLETE")
print(f"  Best epoch:    {best_epoch}")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Best model:    {BEST_MODEL_PATH}")
print(f"  Total time:    {format_time(total_time)}")
print("=" * 60)


Training epochs 1 to 10...
Progress: 28974 train batches + 3473 val batches per epoch
Metrics use PIT-aligned frame/speaker predictions at threshold 0.50.


Epoch 001/10 train:   0%|          | 0/28974 [00:00<?, ?batch/s]

Epoch 001/10 val:   0%|          | 0/3473 [00:00<?, ?batch/s]

Epoch 001/10 train_loss=0.6905 train_acc=0.864 train_f1=0.000 val_loss=0.6849 val_acc=0.870 val_f1=0.000 time=01h 29m 12s total_eta=13h 22m 52s best_val=inf
  Train P/R/F1: 0.001/0.000/0.000 | Val P/R/F1: 0.000/0.000/0.000
  Checkpoint: /content/drive/MyDrive/ami_diarization_v2/checkpoints_head_only/epoch_001.pt
  Best model updated (epoch 1, val=0.6849)


Epoch 002/10 train:   0%|          | 0/28974 [00:00<?, ?batch/s]

# Section 12: Evaluation

> **Note:** Chunk-level DER over preprocessed 10-second chunks, not full pyannote pipeline DER.
> Not directly comparable to published AMI SOTA numbers.

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Annotation, Segment

EVAL_THRESHOLDS = [0.1, 0.2, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]


def match_speakers(pred_np: np.ndarray, target_np: np.ndarray) -> np.ndarray:
    num_speakers = pred_np.shape[1]
    cost = np.zeros((num_speakers, num_speakers))
    for i in range(num_speakers):
        for j in range(num_speakers):
            cost[i, j] = np.sum(np.abs(pred_np[:, i] - target_np[:, j]))
    row_ind, col_ind = linear_sum_assignment(cost)
    matched = np.zeros_like(pred_np)
    for i, j in zip(row_ind, col_ind):
        matched[:, j] = pred_np[:, i]
    return matched


def to_annotation(scores: np.ndarray, threshold: float, frame_duration: float) -> Annotation:
    annotation = Annotation()
    num_frames, num_speakers = scores.shape
    for spk in range(num_speakers):
        in_speech = False
        start     = 0.0
        for f, val in enumerate(scores[:, spk]):
            t = f * frame_duration
            if val > threshold and not in_speech:
                start     = t
                in_speech = True
            elif val <= threshold and in_speech:
                annotation[Segment(start, t)] = f"speaker_{spk}"
                in_speech = False
        if in_speech:
            annotation[Segment(start, num_frames * frame_duration)] = f"speaker_{spk}"
    return annotation


def evaluate_model(eval_model, loader, thresholds, label="model"):
    eval_model.eval()
    results = {}

    for thresh in thresholds:
        all_ders = []
        with torch.no_grad():
            for batch in loader:
                waveforms  = batch["waveforms"].to(device)
                targets    = batch["targets"]
                outputs    = eval_model(waveforms)
                min_frames = min(outputs.shape[1], targets.shape[1])

                # Derive frame duration from actual output length, not hardcoded
                frame_duration = CHUNK_DURATION / min_frames

                pred_np   = torch.sigmoid(outputs[0, :min_frames, :]).cpu().numpy()
                target_np = targets[0, :min_frames, :].numpy()

                matched   = match_speakers(pred_np, target_np)
                hyp       = to_annotation(matched,   thresh, frame_duration)
                ref       = to_annotation(target_np, 0.5,   frame_duration)

                der_metric = DiarizationErrorRate()
                all_ders.append(der_metric(ref, hyp))

        results[thresh] = np.array(all_ders)

    print(f"\n{label} — threshold sweep:")
    best_thresh, best_mean = None, float("inf")
    for thresh, ders in results.items():
        mean_der = np.mean(ders) * 100
        print(
            f"  thresh={thresh:.2f}  "
            f"mean={mean_der:.2f}%  "
            f"median={np.median(ders)*100:.2f}%  "
            f"min={np.min(ders)*100:.2f}%  "
            f"max={np.max(ders)*100:.2f}%"
        )
        if mean_der < best_mean:
            best_mean, best_thresh = mean_der, thresh
    print(f"  Best threshold: {best_thresh:.2f} -> mean DER {best_mean:.2f}%")
    return results


print("Evaluation helpers defined.")

In [ ]:
print("Evaluating pretrained baseline (no fine-tuning)...")
baseline_model   = PyanModel.from_pretrained(MODEL_ID).to(device)
baseline_results = evaluate_model(baseline_model, test_loader, EVAL_THRESHOLDS, label="Pretrained baseline")
del baseline_model

In [ ]:
print(f"Evaluating fine-tuned model from: {BEST_MODEL_PATH}")
finetuned_model = PyanModel.from_pretrained(MODEL_ID).to(device)
finetuned_model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=False, map_location=device))
finetuned_results = evaluate_model(finetuned_model, test_loader, EVAL_THRESHOLDS, label="Fine-tuned model")

In [ ]:
print("\nSide-by-side comparison (mean DER %):")
print(f"{'Threshold':>12}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Delta':>8}")
print("-" * 50)
for thresh in EVAL_THRESHOLDS:
    base = np.mean(baseline_results[thresh]) * 100
    ft   = np.mean(finetuned_results[thresh]) * 100
    d    = ft - base
    sign = "+" if d >= 0 else ""
    print(f"{thresh:>12.2f}  {base:>10.2f}%  {ft:>12.2f}%  {sign}{d:>7.2f}%")